# FACT Reproduction: Spatio-Temporal Accuracy Vid08

This notebook computes per-neuron mASTA and the dataset average for Vid08.


In [1]:
import gc
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)


ROI_SIZE = (128, 64, 64)
MAX_SW_BATCH_SIZE = 32


def _is_cuda_oom(error):
    return isinstance(error, torch.cuda.OutOfMemoryError) or (
        isinstance(error, RuntimeError) and "out of memory" in str(error).lower()
    )


def _release_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_auto_sw_batch_size(
    roi_size=ROI_SIZE,
    max_batch_size=MAX_SW_BATCH_SIZE,
    reserved_ratio=0.05,
    mem_per_patch_mb=705,
    device=None,
):
    """Choose a patch batch from current free VRAM; CPU always uses one patch."""
    del roi_size
    target_device = (
        torch.device(device)
        if device is not None
        else torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    )
    if target_device.type != "cuda" or not torch.cuda.is_available():
        return 1

    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info(target_device)
    safety_ratio = min(max(float(reserved_ratio), 0.0), 0.95)
    usable_bytes = max(0, int(float(free_bytes) * (1.0 - safety_ratio)))
    mem_per_patch = max(1, int(mem_per_patch_mb)) * 1024 * 1024
    estimated_batch = int(usable_bytes // mem_per_patch)
    return max(1, min(int(max_batch_size), estimated_batch))


def _find_release_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        required = (
            candidate / "IO",
            candidate / "ModelInference",
            candidate / "model",
            candidate / "ModelParams" / "FACT_Modelparams.pt",
            candidate / "data" / "STA_Evaluation",
            candidate / "data" / "STA_Evaluation" / "GT",
        )
        if all(path.exists() for path in required):
            return candidate
    raise RuntimeError("Could not locate the FACT-Code-Release directory.")


PROJECT_ROOT = _find_release_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import SimpleITK as sitk
import numpy as np
import scipy.io
import torch
from scipy.sparse import lil_matrix

from IO.Read_tif import load_tiff
from ModelInference.SWInf import sliding_window_inference
from model.TS_Net_change import FACT_Net

### Load data

In [2]:
DATASET_NAME = "Vid08"
DATA_ROOT = PROJECT_ROOT / "data" / "STA_Evaluation"
DATA_PATH = DATA_ROOT / f"{DATASET_NAME}.tiff"
GT_PATH = DATA_ROOT / "GT" / f"{DATASET_NAME}.mat"
MODEL_PATH = PROJECT_ROOT / "ModelParams" / "FACT_Modelparams.pt"

for path in (DATA_PATH, GT_PATH, MODEL_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

In [3]:
input_img = np.asarray(load_tiff(str(DATA_PATH)), dtype=np.float32)
if input_img.ndim != 3:
    raise ValueError(f"Expected a T x H x W video, got {input_img.shape!r}.")

raw_min = float(input_img.min())
raw_max = float(input_img.max())
if not np.isfinite(raw_min) or not np.isfinite(raw_max) or raw_max <= raw_min:
    raise ValueError(f"Cannot normalize input with range [{raw_min}, {raw_max}].")

input_img = (input_img - raw_min) / (raw_max - raw_min)

### Load FACT network

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = FACT_Net(
    img_size=(128, 64, 64),
    in_channels=1,
    out_channels=2,
    init_dim=3,
    drop_rate=0.0,
    attn_drop_rate=0.0,
    use_checkpoint=True,
).to(device)

weight = torch.load(str(MODEL_PATH), map_location="cpu")
model.load_state_dict(weight["state_dict"])
model.eval();

### Network Inference

In [5]:
# Tested configuration: sw_batch_size=32 on an NVIDIA RTX 3090 with 24 GiB VRAM.
sw_batch_size = MAX_SW_BATCH_SIZE


def _run_inference_once(batch_size):
    input_img_tensor = torch.from_numpy(input_img)
    test_inputs = input_img_tensor.unsqueeze(0).unsqueeze(1)
    with torch.no_grad():
        test_outputs = sliding_window_inference(
            test_inputs,
            ROI_SIZE,
            sw_batch_size=int(batch_size),
            predictor=model,
            overlap=[0.6, 0.2, 0.2],
            progress=False,
            mode="constant",
            device=torch.device("cpu"),
            sw_device=device,
        )
        diff = test_outputs[0, 1, :, :, :] - test_outputs[0, 0, :, :, :]
        infer_mask3d_local = (diff > 0).to(dtype=torch.uint8).cpu().numpy()
    del test_outputs, diff, test_inputs, input_img_tensor
    return infer_mask3d_local


while True:
    try:
        infer_mask3d = _run_inference_once(sw_batch_size)
        break
    except RuntimeError as exc:
        if device.type != "cuda" or not _is_cuda_oom(exc):
            raise
        del exc
        _release_cuda_memory()
        if sw_batch_size > 1:
            previous_batch_size = sw_batch_size
            next_batch_size = max(1, previous_batch_size // 2)
            if previous_batch_size == MAX_SW_BATCH_SIZE:
                estimated_batch_size = get_auto_sw_batch_size(
                    roi_size=ROI_SIZE,
                    max_batch_size=MAX_SW_BATCH_SIZE,
                    reserved_ratio=0.05,
                    mem_per_patch_mb=705,
                    device=device,
                )
                next_batch_size = min(next_batch_size, estimated_batch_size)
            sw_batch_size = next_batch_size
            continue

        model = model.to(torch.device("cpu"))
        device = torch.device("cpu")
        _release_cuda_memory()
        sw_batch_size = 1
        infer_mask3d = _run_inference_once(sw_batch_size)
        break

if device.type == "cuda":
    torch.cuda.empty_cache()

infer_mask3d = np.transpose(infer_mask3d, (1, 2, 0))
gc.collect();

## Evaluation

In [6]:
THRESHOLDS = (0.9, 0.8, 0.7, 0.6, 0.5)


def _calculate_sta(mat_path, prediction, t_off):
    t_on = 1.0 - t_off
    mat_data = scipy.io.loadmat(str(mat_path))

    ideal_signalonly = mat_data["ideal_signalonly"].astype(np.int64)
    height, width, neuron_count = ideal_signalonly.shape
    pixel_count = height * width

    gt_masks = lil_matrix((neuron_count, pixel_count), dtype=np.int64)
    for neuron_idx in range(neuron_count):
        gt_masks[neuron_idx] = ideal_signalonly[:, :, neuron_idx].reshape(1, -1)

    pulse_cell = mat_data["pulse_neuron_cell"]
    frame_count = pulse_cell[0, 0].shape[1]
    gt_time = np.zeros((neuron_count, frame_count), dtype=bool)
    for neuron_idx in range(neuron_count):
        gt_time[neuron_idx] = pulse_cell[neuron_idx, 0].astype(bool)

    prediction_height, prediction_width, prediction_frames = prediction.shape
    if (height, width) != (prediction_height, prediction_width):
        raise ValueError(
            "Spatial dimensions differ between the GT masks and the prediction."
        )
    if prediction_frames < frame_count:
        raise ValueError(
            "The prediction contains fewer frames than the GT time annotations."
        )

    predicted_masks = lil_matrix(
        (frame_count, pixel_count), dtype=prediction.dtype
    )
    for frame_idx in range(frame_count):
        predicted_masks[frame_idx] = prediction[:, :, frame_idx].reshape(1, -1)

    overlap = gt_masks.dot(predicted_masks.transpose()).toarray()
    mask_areas = np.asarray(gt_masks.sum(axis=1)).ravel()
    ts_ratio = overlap / mask_areas[:, np.newaxis]

    predicted_on = (ts_ratio >= t_on).astype(bool)
    predicted_off = (ts_ratio <= t_off).astype(bool)
    correct_on = np.logical_and(predicted_on, gt_time)
    correct_off = np.logical_and(predicted_off, ~gt_time)

    return (correct_on.sum(axis=1) + correct_off.sum(axis=1)) / frame_count


sta_values = np.stack(
    [_calculate_sta(GT_PATH, infer_mask3d, t_off) for t_off in THRESHOLDS],
    axis=0,
)
mASTA_all = np.mean(sta_values, axis=0)
dataset_average = float(np.mean(mASTA_all))

print("neuron_idx\tmASTA")
for neuron_idx, value in enumerate(mASTA_all):
    print(f"{neuron_idx}\t{value:.15f}")
print(f"Dataset average\t{dataset_average:.15f}")

neuron_idx	mASTA
0	0.821666666666667
1	0.774333333333333
2	0.606666666666667
3	0.415666666666667
4	0.705666666666667
5	0.614333333333333
6	0.746000000000000
7	0.612333333333333
8	0.271333333333333
9	0.758000000000000
10	0.760000000000000
11	0.746333333333333
12	0.880666666666667
13	0.807333333333333
14	0.737000000000000
15	0.725666666666667
16	0.513666666666667
17	0.420666666666667
18	0.582000000000000
19	0.576000000000000
20	0.504666666666667
21	0.391000000000000
22	0.538666666666667
23	0.502333333333333
24	0.861333333333333
25	0.312666666666667
26	0.653000000000000
27	0.417000000000000
28	0.534333333333333
29	0.797000000000000
30	0.451333333333333
31	0.376000000000000
32	0.802666666666667
33	0.779333333333333
34	0.866000000000000
35	0.501000000000000
36	0.846000000000000
37	0.488666666666667
38	0.829666666666667
39	0.829000000000000
40	0.598000000000000
41	0.590666666666667
42	0.778000000000000
43	0.801666666666667
44	0.741666666666667
45	0.713666666666667
46	0.835000000000000
47	0.7